# 03 — Delta Front and Upper Fan Processing

This notebook preserves the final workflow used to add Delta Front and Upper Fan
observations to the project.

Workflow:

1. Read the desired SuperCam FITS product from each downloaded ZIP.
2. Extract the SOUND `Shot0...ShotN` arrays and concatenate them into a multishot WAV.
3. Analyze the second shot (`Shot1` in zero-based FITS column naming).
4. Calculate the same time- and frequency-domain metrics used for the Crater Floor.
5. Save accepted and flagged observations separately.
6. Append the 108 automatically accepted Delta/Upper Fan observations to the
   185-row Crater Floor dataset to create the final 293-row dataset.

The source workflow attempted 225 FITS products. The retained concatenation log
records 221 successful WAV conversions. The final accepted-results CSV contains
108 observations (53 Delta, 55 Upper Fan).

In [ ]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import csv

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from astropy.io import fits
from scipy.io import wavfile
from scipy.stats import linregress
from scipy.signal import find_peaks, windows

## Paths

Raw ZIP archives are read from `data/raw/delta_upper_fan_zips/`.
Concatenated multishot WAVs are written to `data/intermediate/concatenated_multishot_wav/`.
QC logs and accepted/flagged processing tables are written to `data/qc/`.
The final combined 293-row dataset is written to `data/processed/`.


In [ ]:
PROJECT_ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")

# Downloaded ZIP archives from the PDS / Mars Analyst Notebook.
RAW_ZIP_ROOT = PROJECT_ROOT / "data" / "raw" / "delta_upper_fan_zips"

ZIP_FOLDERS = {
    "Delta": RAW_ZIP_ROOT / "Delta",
    "Upper Fan": RAW_ZIP_ROOT / "Upper Fan",
}

# Intermediate concatenated multishot WAV products.
OUTPUT_ROOT = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "concatenated_multishot_wav"
)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

FS = 100_000

# Metric-processing output.
root_folder = OUTPUT_ROOT

unit_folders = {
    "Delta": root_folder / "Delta",
    "Upper Fan": root_folder / "Upper Fan",
}

QC_DIR = PROJECT_ROOT / "data" / "qc"
QC_DIR.mkdir(parents=True, exist_ok=True)

flagged_plot_folder = (
    PROJECT_ROOT
    / "figures"
    / "qc"
    / "delta_upper_fan_flagged"
)
flagged_plot_folder.mkdir(parents=True, exist_ok=True)

# Final merge inputs/outputs.
meta_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "LIBS_acoustic_meta_sheet_v3_185.csv"
)

accepted_path = (
    QC_DIR
    / "LIBS_acoustic_accepted_results.csv"
)

final_output_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "LIBS_acoustic_meta_sheet_v4_293.csv"
)

final_output_path.parent.mkdir(parents=True, exist_ok=True)

print("ZIP root:", RAW_ZIP_ROOT.resolve())
print("Concatenated WAV root:", OUTPUT_ROOT.resolve())
print("QC output:", QC_DIR.resolve())
print("Final combined dataset:", final_output_path.resolve())

## 1. FITS → concatenated multishot WAV

The desired FITS products end in `01p02.fits`. The SOUND extension contains
`Shot0`, `Shot1`, ... columns. The original workflow concatenated those shot
arrays in numerical order and wrote them as float32 WAV files at 100 kHz.

This intermediate WAV step is what the final Delta/Upper Fan metric-processing
code actually used.

In [ ]:
# ============================================================
# HELPERS

# ============================================================

def shot_number(column_name):
    """Return the integer in Shot0, Shot1, etc."""
    return int(column_name.lower().replace("shot", ""))


def find_desired_fits(zip_file):
    """
    Find FITS products ending in 01p02.fits inside one ZIP.

    Returns every match because some AEGIS ZIPs may contain
    multiple desired FITS files.
    """
    names = [
        name for name in zip_file.namelist()
        if name.lower().endswith("01p02.fits")
    ]

    return sorted(names)


def read_concatenated_sound(zip_file, fits_name):
    """
    Read SOUND/Shot0...ShotN from a FITS inside an open ZIP
    and concatenate the raw shot arrays in numerical order.
    """

    fits_bytes = zip_file.read(fits_name)

    with fits.open(BytesIO(fits_bytes), memmap=False) as hdul:

        if "SOUND" not in hdul:
            raise KeyError("SOUND extension not found")

        sound = hdul["SOUND"].data

        if sound is None:
            raise ValueError("SOUND extension contains no data")

        column_names = list(sound.columns.names)

        shot_columns = [
            name for name in column_names
            if name.lower().startswith("shot")
            and name[4:].isdigit()
        ]

        if not shot_columns:
            raise ValueError("No Shot columns found in SOUND")

        shot_columns.sort(key=shot_number)

        shots = []

        for column in shot_columns:
            shot = np.asarray(sound[column], dtype=np.float32).reshape(-1)

            if not np.all(np.isfinite(shot)):
                raise ValueError(f"{column} contains NaN or infinite values")

            shots.append(shot)

        waveform = np.concatenate(shots)

    return waveform, shot_columns


def make_output_name(fits_name):
    """
    Convert the FITS product name to a WAV filename while
    preserving its original identifying information.
    """
    return Path(fits_name).name[:-5] + ".wav"


# ============================================================
# PROCESS ALL ZIP FILES
# ============================================================

log_rows = []

for unit, zip_folder in ZIP_FOLDERS.items():

    unit_wav_output_folder = OUTPUT_ROOT / unit
    unit_wav_output_folder.mkdir(parents=True, exist_ok=True)

    zip_paths = sorted(zip_folder.glob("*.zip"))

    print("\n" + "=" * 80)
    print(f"{unit}: {len(zip_paths)} ZIP files")
    print("=" * 80)

    for zip_path in zip_paths:

        try:
            with ZipFile(zip_path, "r") as zf:

                fits_names = find_desired_fits(zf)

                if not fits_names:
                    raise FileNotFoundError(
                        "No file ending in 01p02.fits found"
                    )

                for fits_name in fits_names:

                    try:
                        waveform, shot_columns = read_concatenated_sound(
                            zf,
                            fits_name
                        )

                        output_name = make_output_name(fits_name)
                        wav_output_path = unit_wav_output_folder / output_name

                        # Preserve the FITS floating-point waveform.
                        # scipy writes this as a 32-bit float WAV.
                        wavfile.write(
                            wav_output_path,
                            FS,
                            waveform.astype(np.float32)
                        )

                        duration = len(waveform) / FS

                        print(
                            f"OK | {zip_path.name} | "
                            f"{Path(fits_name).name} | "
                            f"{len(shot_columns)} shots | "
                            f"{len(waveform)} samples | "
                            f"{duration:.3f} s"
                        )

                        log_rows.append({
                            "Unit": unit,
                            "ZIP": zip_path.name,
                            "FITS": Path(fits_name).name,
                            "WAV": output_name,
                            "Status": "OK",
                            "Number of shots": len(shot_columns),
                            "Samples per shot": (
                                len(waveform) // len(shot_columns)
                            ),
                            "Total samples": len(waveform),
                            "Duration (s)": duration,
                            "Error": "",
                        })

                    except Exception as exc:
                        print(
                            f"ERROR | {zip_path.name} | "
                            f"{Path(fits_name).name} | {exc}"
                        )

                        log_rows.append({
                            "Unit": unit,
                            "ZIP": zip_path.name,
                            "FITS": Path(fits_name).name,
                            "WAV": "",
                            "Status": "ERROR",
                            "Number of shots": "",
                            "Samples per shot": "",
                            "Total samples": "",
                            "Duration (s)": "",
                            "Error": str(exc),
                        })

        except Exception as exc:
            print(f"ZIP ERROR | {zip_path.name} | {exc}")

            log_rows.append({
                "Unit": unit,
                "ZIP": zip_path.name,
                "FITS": "",
                "WAV": "",
                "Status": "ZIP ERROR",
                "Number of shots": "",
                "Samples per shot": "",
                "Total samples": "",
                "Duration (s)": "",
                "Error": str(exc),
            })


# ============================================================
# SAVE LOG
# ============================================================

log_path = QC_DIR / "concatenation_log.csv"

fieldnames = [
    "Unit",
    "ZIP",
    "FITS",
    "WAV",
    "Status",
    "Number of shots",
    "Samples per shot",
    "Total samples",
    "Duration (s)",
    "Error",
]

with open(log_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(log_rows)


# ============================================================
# SUMMARY
# ============================================================

ok_count = sum(row["Status"] == "OK" for row in log_rows)
error_count = len(log_rows) - ok_count

print("\n" + "=" * 80)
print("FINISHED")
print("=" * 80)
print(f"WAVs created: {ok_count}")
print(f"Errors:       {error_count}")
print(f"Output:       {OUTPUT_ROOT}")
print(f"Log:          {log_path}")

## 2. Process the second shot

In the concatenated WAVs each shot contains 6000 samples (0.060 s at 100 kHz).
The workflow analyzes the second shot, which occupies 0.060–0.120 s in the
concatenated waveform.

The onset detector, EDC fit, C2 calculation, and FFT metrics below are retained
from the final `ZIP PROCESSING.ipynb` batch-processing block.

In [ ]:
# ============================================================
# GENERAL SETTINGS

# ============================================================

show_all_detection_plots = False
save_flagged_detection_plots = True

expected_sample_rate = 100_000
shot_samples = 6000
shot_duration = shot_samples / expected_sample_rate

# Concatenated WAV structure:
#
# Shot0: 0.000–0.060 s
# Shot1: 0.060–0.120 s  <- analyze this shot
# Shot2: 0.120–0.180 s
#
second_shot_start = 0.060
expected_shot_time = 0.074
next_shot_start = 0.120

response_window = 0.010

fit_db_top = -7
fit_db_bottom = -15

t_c = 0.002


# ============================================================
# ONSET-DETECTION SETTINGS
# Same settings as crater-floor pipeline
# ============================================================

search_half_width = 0.005
noise_window = 0.003

smooth_window_s = 0.00005
threshold_sigma = 4

min_peak_distance_s = 0.010
prominence_sigma = 3
height_sigma = 4

backtrack_window_s = 0.003
backtrack_noise_s = 0.005
backtrack_sigma = 4

min_run_s = 0.00005

# Same crater-floor review threshold
flag_lag_ms = 0.4


# ============================================================
# FFT SETTINGS
# ============================================================

usable_band = (1000, 50000)
low_band = (1000, 10000)
high_band = (10000, 30000)


# ============================================================
# EMPTY FFT OUTPUT
# ============================================================

def empty_fft_metrics():
    return {
        "Spectral Centroid (Hz)": np.nan,
        "Spectral Bandwidth (Hz)": np.nan,
        "Peak Frequency (Hz)": np.nan,
        "Rolloff 85% (Hz)": np.nan,
        "Low Power 1-10 kHz": np.nan,
        "High Power 10-30 kHz": np.nan,
        "High/Low Ratio": np.nan,
        "High Frequency Fraction": np.nan,
        "Total FFT Power": np.nan,
    }


# ============================================================
# FFT METRICS
# ============================================================

def compute_fft_metrics(segment, fs):
    """
    Compute the same frequency-domain metrics used in the
    crater-floor pipeline.
    """

    segment = np.asarray(segment, dtype=np.float64)

    if len(segment) < 2:
        return empty_fft_metrics()

    segment = segment - np.mean(segment)

    hann = windows.hann(len(segment))
    seg_w = segment * hann

    freqs = np.fft.rfftfreq(
        len(seg_w),
        d=1 / fs,
    )

    X = np.fft.rfft(seg_w)
    power = np.abs(X) ** 2

    usable_mask = (
        (freqs >= usable_band[0])
        & (freqs <= usable_band[1])
    )

    f = freqs[usable_mask]
    p = power[usable_mask]

    if len(p) == 0 or np.sum(p) <= 0:
        return empty_fft_metrics()

    p_sum = np.sum(p)

    centroid = np.sum(f * p) / p_sum

    bandwidth = np.sqrt(
        np.sum(
            ((f - centroid) ** 2) * p
        ) / p_sum
    )

    peak_freq = f[np.argmax(p)]

    cumulative = np.cumsum(p)

    rolloff_candidates = np.where(
        cumulative >= 0.85 * p_sum
    )[0]

    if len(rolloff_candidates) > 0:
        rolloff_85 = f[rolloff_candidates[0]]
    else:
        rolloff_85 = np.nan

    low_mask = (
        (freqs >= low_band[0])
        & (freqs < low_band[1])
    )

    high_mask = (
        (freqs >= high_band[0])
        & (freqs < high_band[1])
    )

    low_power = np.sum(power[low_mask])
    high_power = np.sum(power[high_mask])
    total_power = np.sum(power)

    if low_power > 0:
        high_low_ratio = high_power / low_power
    else:
        high_low_ratio = np.nan

    if total_power > 0:
        high_freq_fraction = high_power / total_power
    else:
        high_freq_fraction = np.nan

    return {
        "Spectral Centroid (Hz)": centroid,
        "Spectral Bandwidth (Hz)": bandwidth,
        "Peak Frequency (Hz)": peak_freq,
        "Rolloff 85% (Hz)": rolloff_85,
        "Low Power 1-10 kHz": low_power,
        "High Power 10-30 kHz": high_power,
        "High/Low Ratio": high_low_ratio,
        "High Frequency Fraction": high_freq_fraction,
        "Total FFT Power": total_power,
    }


# ============================================================
# LOAD WAV
# ============================================================

def load_wav_mono(wav_path):
    """
    Load a WAV file and convert it to mono float64.
    """

    fs, x = wavfile.read(wav_path)

    x = np.asarray(x)

    if x.ndim > 1:
        x = x.astype(np.float64).mean(axis=1)
    else:
        x = x.astype(np.float64)

    return fs, x.reshape(-1)


# ============================================================
# PROCESS ONE WAV
# ============================================================

def process_one_wav(wav_path, unit):
    """
    Run the original crater-floor onset and metric pipeline
    on Shot1 of one concatenated WAV.
    """

    fs, x = load_wav_mono(wav_path)

    total_samples = len(x)
    duration_s = total_samples / fs

    if fs != expected_sample_rate:
        raise ValueError(
            f"Expected {expected_sample_rate} Hz, "
            f"but found {fs} Hz."
        )

    if total_samples % shot_samples != 0:
        raise ValueError(
            f"{total_samples} samples is not divisible "
            f"by {shot_samples} samples per shot."
        )

    number_of_shots = total_samples // shot_samples

    if number_of_shots < 2:
        raise ValueError(
            "Recording contains fewer than two shots."
        )

    # Same preprocessing as crater-floor pipeline.
    x = x - np.mean(x)

    abs_x = np.abs(x)

    smooth_n = max(
        1,
        int(round(smooth_window_s * fs)),
    )

    kernel = np.ones(smooth_n) / smooth_n

    env = np.convolve(
        abs_x,
        kernel,
        mode="same",
    )

    # --------------------------------------------------------
    # EXPECTED-ONSET SEARCH REGION
    # --------------------------------------------------------

    search_start = (
        expected_shot_time
        - search_half_width
    )

    search_end = (
        expected_shot_time
        + search_half_width
    )

    isearch0 = max(
        0,
        int(round(search_start * fs)),
    )

    isearch1 = min(
        len(x),
        int(round(search_end * fs)),
    )

    # Baseline immediately before the search region.
    inoise1 = isearch0

    inoise0 = max(
        0,
        int(
            round(
                (search_start - noise_window) * fs
            )
        ),
    )

    noise_env = env[inoise0:inoise1]

    if len(noise_env) == 0:
        raise ValueError(
            "Noise window contains no samples."
        )

    noise_mean = np.mean(noise_env)
    noise_std = np.std(noise_env)

    threshold = (
        noise_mean
        + threshold_sigma * noise_std
    )

    search_env = env[isearch0:isearch1]

    min_peak_distance = max(
        1,
        int(round(min_peak_distance_s * fs)),
    )

    min_prominence = (
        prominence_sigma * noise_std
    )

    min_height = (
        noise_mean
        + height_sigma * noise_std
    )

    peaks, props = find_peaks(
        search_env,
        height=min_height,
        prominence=min_prominence,
        distance=min_peak_distance,
    )

    found_peak = len(peaks) > 0

    flagged = False
    flag_reason = ""
    lag_ms = np.nan

    onset_index = None
    peak_index = None

    response_start = np.nan
    response_stop = np.nan
    peak_time = np.nan
    onset_threshold = np.nan

    slope = np.nan
    intercept = np.nan
    r_value = np.nan
    drop_time = np.nan
    C2 = np.nan

    segment = None
    segment_t = None
    edc_db = None
    fit_mask = None
    fit_line = None

    fft_metrics = empty_fft_metrics()

    # --------------------------------------------------------
    # PEAK FOUND
    # --------------------------------------------------------

    if found_peak:
        # Same crater-floor rule: use first qualifying peak.
        best_peak_local = peaks[0]

        peak_index = (
            isearch0
            + best_peak_local
        )

        peak_time = peak_index / fs

        # ----------------------------------------------------
        # BACKTRACK FROM PEAK TO ONSET
        # ----------------------------------------------------

        backtrack_samples = int(
            round(backtrack_window_s * fs)
        )

        second_shot_start_index = int(
            round(second_shot_start * fs)
        )

        back_start = max(
            second_shot_start_index,
            peak_index - backtrack_samples,
        )

        backtrack_noise_samples = int(
            round(backtrack_noise_s * fs)
        )

        noise_back_start = max(
            0,
            back_start - backtrack_noise_samples,
        )

        noise_back_end = back_start

        local_noise = env[
            noise_back_start:noise_back_end
        ]

        if len(local_noise) > 0:
            local_noise_mean = np.mean(
                local_noise
            )

            local_noise_std = np.std(
                local_noise
            )

            onset_threshold = (
                local_noise_mean
                + backtrack_sigma
                * local_noise_std
            )

        else:
            onset_threshold = threshold

        search_back_env = env[
            back_start:peak_index
        ]

        above = (
            search_back_env
            > onset_threshold
        )

        min_run_samples = max(
            1,
            int(round(min_run_s * fs)),
        )

        for i in range(
            len(above)
            - min_run_samples
            + 1
        ):
            if np.all(
                above[
                    i:i + min_run_samples
                ]
            ):
                onset_index = back_start + i
                break

        # Same behavior as original code:
        # use peak as onset but flag the file.
        if onset_index is None:
            onset_index = peak_index
            flagged = True
            flag_reason = (
                "NO CLEAN ONSET BEFORE PEAK"
            )

        response_start = onset_index / fs

        response_stop = (
            response_start
            + response_window
        )

        lag_ms = (
            peak_index - onset_index
        ) / fs * 1000

        if lag_ms > flag_lag_ms:
            flagged = True

            if flag_reason:
                flag_reason += (
                    " | LONG PEAK-ONSET LAG"
                )
            else:
                flag_reason = (
                    "LONG PEAK-ONSET LAG"
                )

        # Prevent response window from entering Shot2.
        if response_stop > next_shot_start:
            response_stop = next_shot_start

            flagged = True

            if flag_reason:
                flag_reason += (
                    " | WINDOW TRUNCATED"
                )
            else:
                flag_reason = (
                    "WINDOW TRUNCATED"
                )

        # ----------------------------------------------------
        # EXTRACT RESPONSE SEGMENT
        # ----------------------------------------------------

        i0 = onset_index

        i1 = min(
            len(x),
            int(round(response_stop * fs)),
        )

        segment = x[i0:i1]

        segment_t = (
            np.arange(len(segment)) / fs
        )

        # ----------------------------------------------------
        # ENERGY DECAY CURVE
        # ----------------------------------------------------

        if len(segment) > 1:
            energy = segment ** 2

            edc = np.cumsum(
                energy[::-1]
            )[::-1]

            edc_max = np.max(edc)

            if edc_max > 0:
                edc_norm = edc / edc_max

                edc_db = 10 * np.log10(
                    edc_norm + 1e-20
                )

                fit_mask = (
                    (edc_db <= fit_db_top)
                    & (edc_db >= fit_db_bottom)
                )

                if np.sum(fit_mask) >= 2:
                    (
                        slope,
                        intercept,
                        r_value,
                        p_value,
                        std_err,
                    ) = linregress(
                        segment_t[fit_mask],
                        edc_db[fit_mask],
                    )

                    fit_line = (
                        slope * segment_t
                        + intercept
                    )

                    db_drop = abs(
                        fit_db_bottom
                        - fit_db_top
                    )

                    if slope != 0:
                        drop_time = (
                            db_drop
                            / abs(slope)
                        )

                else:
                    fit_line = np.full_like(
                        edc_db,
                        np.nan,
                    )

                # --------------------------------------------
                # C2
                # --------------------------------------------

                i_c = int(round(t_c * fs))

                i_c = min(
                    i_c,
                    len(energy),
                )

                early_energy = np.sum(
                    energy[:i_c]
                )

                late_energy = np.sum(
                    energy[i_c:]
                )

                if (
                    early_energy > 0
                    and late_energy > 0
                ):
                    C2 = 10 * np.log10(
                        early_energy
                        / late_energy
                    )

            # ------------------------------------------------
            # FFT METRICS
            # ------------------------------------------------

            fft_metrics = compute_fft_metrics(
                segment,
                fs,
            )

    # --------------------------------------------------------
    # NO PEAK FOUND
    # --------------------------------------------------------

    else:
        flagged = True
        flag_reason = "NO PEAK"

    if not flagged:
        status = "OK"
    else:
        status = flag_reason

    row = {
        "Unit": unit,
        "File Name": wav_path.name,
        "Path": str(wav_path),

        "Status": status,
        "Flagged": flagged,

        "Sample Rate (Hz)": fs,
        "Samples": total_samples,
        "Duration (s)": duration_s,
        "Number of Shots": number_of_shots,
        "Observation Type":
            f"{number_of_shots}-shot",

        "Search Start (s)": search_start,
        "Search End (s)": search_end,

        "Onset Sample": (
            onset_index
            if onset_index is not None
            else np.nan
        ),

        "Onset (s)": response_start,

        "Onset Within Shot1 (ms)": (
            (
                response_start
                - second_shot_start
            ) * 1000
            if np.isfinite(response_start)
            else np.nan
        ),

        "Peak Sample": (
            peak_index
            if peak_index is not None
            else np.nan
        ),

        "Peak Time (s)": peak_time,
        "Peak-Onset Lag (ms)": lag_ms,

        "Response Stop (s)": response_stop,
        "Response Duration (ms)": (
            (
                response_stop
                - response_start
            ) * 1000
            if (
                np.isfinite(response_start)
                and np.isfinite(response_stop)
            )
            else np.nan
        ),

        "Slope (dB/s)": slope,

        "Decay Rate (-Slope) (dB/s)": (
            -slope
            if np.isfinite(slope)
            else np.nan
        ),

        "R^2": (
            r_value ** 2
            if np.isfinite(r_value)
            else np.nan
        ),

        "Drop Time (s)": drop_time,
        "C2 (dB)": C2,

        **fft_metrics,
    }

    plot_data = {
        "x": x,
        "env": env,
        "fs": fs,

        "search_start": search_start,
        "search_end": search_end,
        "isearch0": isearch0,
        "isearch1": isearch1,
        "back_start": (
            back_start
            if found_peak
            else isearch0
        ),

        "threshold": threshold,
        "onset_threshold": onset_threshold,

        "found_peak": found_peak,
        "onset_index": onset_index,
        "peak_index": peak_index,

        "segment_t": segment_t,
        "edc_db": edc_db,
        "fit_mask": fit_mask,
        "fit_line": fit_line,
    }

    return row, plot_data


# ============================================================
# DETECTION PLOT
# ============================================================

def make_detection_plot(
    wav_path,
    unit,
    row,
    plot_data,
    save_path=None,
):
    """
    Create the same three-part review plot:
    waveform, envelope, and EDC.
    """

    x = plot_data["x"]
    env = plot_data["env"]
    fs = plot_data["fs"]

    isearch0 = plot_data["isearch0"]
    isearch1 = plot_data["isearch1"]
    back_start = plot_data["back_start"]

    search_start = plot_data["search_start"]
    search_end = plot_data["search_end"]

    threshold = plot_data["threshold"]
    onset_threshold = plot_data[
        "onset_threshold"
    ]

    found_peak = plot_data["found_peak"]
    onset_index = plot_data["onset_index"]
    peak_index = plot_data["peak_index"]

    segment_t = plot_data["segment_t"]
    edc_db = plot_data["edc_db"]
    fit_mask = plot_data["fit_mask"]
    fit_line = plot_data["fit_line"]

    full_t = np.arange(len(x)) / fs

    # Show the full possible onset-backtracking region in QA plots,
    # not only the narrower peak-search window.
    iplot0 = min(back_start, isearch0)
    iplot1 = isearch1

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(11, 9),
    )

    # --------------------------------------------------------
    # WAVEFORM
    # --------------------------------------------------------

    axes[0].plot(
        full_t[iplot0:iplot1],
        x[iplot0:iplot1],
        linewidth=0.8,
    )

    if onset_index is not None:
        axes[0].axvline(
            onset_index / fs,
            linestyle="--",
            label="Onset",
        )

    if peak_index is not None:
        axes[0].axvline(
            peak_index / fs,
            linestyle=":",
            label="Peak",
        )

    axes[0].axvline(
        search_start,
        linestyle="-.",
        linewidth=0.8,
        label="Peak search start",
    )

    axes[0].axvline(
        search_end,
        linestyle="-.",
        linewidth=0.8,
        label="Peak search end",
    )

    axes[0].set_title(
        "Waveform: Backtracking + Peak-Search Region"
    )

    axes[0].set_xlabel("Time (s)")
    axes[0].set_ylabel("Amplitude")

    if onset_index is not None:
        axes[0].legend()

    # --------------------------------------------------------
    # ENVELOPE
    # --------------------------------------------------------

    axes[1].plot(
        full_t[iplot0:iplot1],
        env[iplot0:iplot1],
        linewidth=0.9,
        label="Smoothed envelope",
    )

    axes[1].axhline(
        threshold,
        linestyle=":",
        label="Peak threshold",
    )

    if np.isfinite(onset_threshold):
        axes[1].axhline(
            onset_threshold,
            linestyle="--",
            label="Onset threshold",
        )

    if onset_index is not None:
        axes[1].axvline(
            onset_index / fs,
            linestyle="--",
        )

    if peak_index is not None:
        axes[1].axvline(
            peak_index / fs,
            linestyle=":",
        )

    axes[1].axvline(
        search_start,
        linestyle="-.",
        linewidth=0.8,
        label="Peak search start",
    )

    axes[1].axvline(
        search_end,
        linestyle="-.",
        linewidth=0.8,
        label="Peak search end",
    )

    axes[1].set_title(
        "Smoothed Envelope: Backtracking + Peak Search"
    )

    axes[1].set_xlabel("Time (s)")
    axes[1].set_ylabel("Envelope")
    axes[1].legend()

    # --------------------------------------------------------
    # EDC
    # --------------------------------------------------------

    if (
        found_peak
        and segment_t is not None
        and edc_db is not None
    ):
        axes[2].plot(
            segment_t,
            edc_db,
            label="EDC",
        )

        if (
            fit_mask is not None
            and np.sum(fit_mask) >= 2
        ):
            axes[2].plot(
                segment_t,
                fit_line,
                "--",
                label="-7 to -15 dB fit",
            )

        axes[2].axhline(
            fit_db_top,
            linestyle=":",
            linewidth=0.8,
        )

        axes[2].axhline(
            fit_db_bottom,
            linestyle=":",
            linewidth=0.8,
        )

        axes[2].set_title(
            "Energy Decay Curve"
        )

        axes[2].set_xlabel(
            "Time after onset (s)"
        )

        axes[2].set_ylabel("EDC (dB)")
        axes[2].legend()

    else:
        axes[2].text(
            0.5,
            0.5,
            "No onset detected",
            horizontalalignment="center",
            verticalalignment="center",
            transform=axes[2].transAxes,
        )

        axes[2].set_title(
            "Energy Decay Curve"
        )

    fig.suptitle(
        f"{unit} | {wav_path.name}\n"
        f"Status: {row['Status']}",
        fontsize=11,
    )

    plt.tight_layout()

    if save_path is not None:
        plt.savefig(
            save_path,
            dpi=170,
            bbox_inches="tight",
        )

    if show_all_detection_plots:
        plt.show()
    else:
        plt.close(fig)


# ============================================================
# PROCESS ALL FILES
# ============================================================

results = []

for unit, wav_folder in unit_folders.items():
    files = sorted(
        wav_folder.glob("*.wav")
    )

    print("\n" + "=" * 90)
    print(f"{unit}: {len(files)} WAV files")
    print("=" * 90)

    for file_number, wav_path in enumerate(
        files,
        start=1,
    ):
        try:
            row, plot_data = process_one_wav(
                wav_path,
                unit,
            )

        except Exception as exc:
            row = {
                "Unit": unit,
                "File Name": wav_path.name,
                "Path": str(wav_path),
                "Status": "ERROR",
                "Flagged": True,
                "Error": str(exc),
            }

            plot_data = None

        results.append(row)

        lag = row.get(
            "Peak-Onset Lag (ms)",
            np.nan,
        )

        onset = row.get(
            "Onset (s)",
            np.nan,
        )

        if np.isfinite(onset):
            onset_text = f"{onset:.6f} s"
        else:
            onset_text = "None"

        if np.isfinite(lag):
            lag_text = f"{lag:.3f} ms"
        else:
            lag_text = "None"

        print(
            f"[{file_number:03d}/{len(files):03d}] "
            f"{row['Status']:<30} | "
            f"onset={onset_text:<12} | "
            f"lag={lag_text:<10} | "
            f"{wav_path.name}"
        )

        # Save QA plots only for flagged files.
        if (
            save_flagged_detection_plots
            and row["Flagged"]
            and plot_data is not None
        ):
            safe_status = (
                row["Status"]
                .replace(" ", "_")
                .replace("|", "_")
                .replace("/", "_")
            )

            plot_name = (
                f"{unit}_{safe_status}_"
                f"{wav_path.stem}.png"
            )

            make_detection_plot(
                wav_path=wav_path,
                unit=unit,
                row=row,
                plot_data=plot_data,
                save_path=(
                    flagged_plot_folder
                    / plot_name
                ),
            )

        elif (
            show_all_detection_plots
            and plot_data is not None
        ):
            make_detection_plot(
                wav_path=wav_path,
                unit=unit,
                row=row,
                plot_data=plot_data,
            )


# ============================================================
# EXPORT RESULTS
# ============================================================

results_df = pd.DataFrame(results)

all_results_path = (
    QC_DIR
    / "LIBS_acoustic_all_results.csv"
)

accepted_results_path = (
    QC_DIR
    / "LIBS_acoustic_accepted_results.csv"
)

flagged_results_path = (
    QC_DIR
    / "LIBS_acoustic_flagged_results.csv"
)

results_df.to_csv(
    all_results_path,
    index=False,
)

accepted_df = results_df[
    results_df["Flagged"] == False
].copy()

accepted_df.to_csv(
    accepted_results_path,
    index=False,
)

flagged_df = results_df[
    results_df["Flagged"] == True
].copy()

flagged_df.to_csv(
    flagged_results_path,
    index=False,
)


# Separate unit files
for unit in unit_folders:
    unit_df = results_df[
        results_df["Unit"] == unit
    ].copy()

    unit_output_path = (
        QC_DIR
        / f"{unit.replace(' ', '_')}_LIBS_acoustic_results.csv"
    )

    unit_df.to_csv(
        unit_output_path,
        index=False,
    )


# ============================================================
# SUMMARIES
# ============================================================

print("\n" + "=" * 90)
print("OVERALL STATUS COUNTS")
print("=" * 90)

print(
    results_df["Status"]
    .value_counts(dropna=False)
)


print("\n" + "=" * 90)
print("STATUS BY UNIT")
print("=" * 90)

print(
    pd.crosstab(
        results_df["Unit"],
        results_df["Status"],
    )
)


print("\n" + "=" * 90)
print("FLAGGED COUNTS BY UNIT")
print("=" * 90)

print(
    pd.crosstab(
        results_df["Unit"],
        results_df["Flagged"],
    )
)


print("\n" + "=" * 90)
print("ACCEPTED ONSET STATISTICS")
print("=" * 90)

if len(accepted_df) > 0:
    summary_columns = [
        "Onset (s)",
        "Onset Within Shot1 (ms)",
        "Peak-Onset Lag (ms)",
        "Slope (dB/s)",
        "R^2",
        "Drop Time (s)",
        "C2 (dB)",
        "Spectral Centroid (Hz)",
        "Spectral Bandwidth (Hz)",
        "Peak Frequency (Hz)",
        "Rolloff 85% (Hz)",
        "High/Low Ratio",
        "High Frequency Fraction",
    ]

    available_columns = [
        column
        for column in summary_columns
        if column in accepted_df.columns
    ]

    print(
        accepted_df[
            available_columns
        ]
        .describe()
        .round(4)
    )

else:
    print(
        "No files were automatically accepted."
    )


print("\n" + "=" * 90)
print("OUTPUT FILES")
print("=" * 90)

print(f"All results:      {all_results_path}")
print(f"Accepted results: {accepted_results_path}")
print(f"Flagged results:  {flagged_results_path}")
print(f"Flagged plots:    {flagged_plot_folder}")

## 3. Merge accepted observations with the Crater Floor dataset

Only rows with `Flagged == False` were included in the final regional dataset.
The retained accepted-results file contains 108 observations. Those rows are
appended to the 185-row Crater Floor dataset and labeled by region/provenance.

In [ ]:
# ------------------------------------------------------------
# Load files

# ------------------------------------------------------------

meta = pd.read_csv(meta_path)
accepted = pd.read_csv(accepted_path)

print("Existing meta rows:", len(meta))
print("Accepted new rows:", len(accepted))

# ------------------------------------------------------------
# Confirm expected new-unit labels
# ------------------------------------------------------------

if "Unit" not in accepted.columns:
    raise KeyError(
        "The accepted-results file does not contain a 'Unit' column."
    )

print("\nAccepted Unit counts:")
print(accepted["Unit"].value_counts(dropna=False))

unexpected_units = set(
    accepted["Unit"].dropna().unique()
) - {"Delta", "Upper Fan"}

if unexpected_units:
    raise ValueError(
        f"Unexpected Unit values found: {unexpected_units}"
    )

# ------------------------------------------------------------
# Fill geological classification for new rows
# ------------------------------------------------------------

accepted["Formation"] = accepted["Unit"]
accepted["Member"] = pd.NA

# Optional broad region column
if "Region" not in meta.columns:
    meta["Region"] = "Crater Floor"
else:
    meta["Region"] = meta["Region"].fillna(
        "Crater Floor"
    )

accepted["Region"] = accepted["Unit"]

# Optional provenance columns
if "Dataset Source" not in meta.columns:
    meta["Dataset Source"] = "Crater Floor"

accepted["Dataset Source"] = accepted["Unit"]

if "Processing Version" not in meta.columns:
    meta["Processing Version"] = pd.NA

accepted["Processing Version"] = (
    "Short-clip find_peaks backtracking v2"
)

# ------------------------------------------------------------
# Match all columns
# ------------------------------------------------------------

all_columns = list(meta.columns)

for col in accepted.columns:
    if col not in all_columns:
        all_columns.append(col)

meta = meta.reindex(columns=all_columns)
accepted = accepted.reindex(columns=all_columns)

# ------------------------------------------------------------
# Concatenate
# ------------------------------------------------------------

combined = pd.concat(
    [meta, accepted],
    ignore_index=True
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_total = len(meta) + len(accepted)

if len(combined) != expected_total:
    raise RuntimeError(
        "Unexpected row-count change during concatenation."
    )

print("\nCombined rows:", len(combined))

print("\nFormation counts:")
print(
    combined["Formation"]
    .fillna("Missing")
    .value_counts()
)

print("\nRegion counts:")
print(
    combined["Region"]
    .fillna("Missing")
    .value_counts()
)

print("\nNew rows by Formation:")
print(
    combined.tail(len(accepted))["Formation"]
    .value_counts(dropna=False)
)

print(
    "\nExact duplicate rows:",
    combined.duplicated().sum()
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

combined.to_csv(final_output_path, index=False)

print("\nSaved new master metasheet to:")
print(final_output_path)

## Expected result

With the original ZIP collection and final processing settings, the retained
project files indicate:

- 225 FITS products attempted during concatenation
- 221 successfully converted/processed WAV observations
- 108 automatically accepted observations
- 53 Delta observations
- 55 Upper Fan observations
- 185 Crater Floor + 108 new observations = 293 total rows

Flagged observations remain available in `LIBS_acoustic_flagged_results.csv`
for any future manual review.